1

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, avg, count, desc, round, when

spark = SparkSession.builder \
    .appName("CrimeAnalysisIndia") \
    .getOrCreate()

print("Spark Version:", spark.version)

Spark Version: 4.0.2


2- load ds

In [3]:
from google.colab import files
uploaded = files.upload()  # upload dataset.csv

df = spark.read.csv("dataset.csv", header=True, inferSchema=True)

df.printSchema()
df.show(3)
print("Total Records:", df.count())   # 9017
print("Total Columns:", len(df.columns))  # 33

KeyboardInterrupt: 

3 - Data exploration

In [ ]:
# ── Data Overview ────────────────────────────────────────────────────
print("Dataset Shape")
print(f"Rows: {df.count()}, Columns: {len(df.columns)}")

print("\nSample Records")
df.show(5)

print("\nBasic Statistics")
df.select("MURDER", "RAPE", "THEFT", "ROBBERY",
          "TOTAL IPC CRIMES").describe().show()

print("\n Null Value Check")
from pyspark.sql.functions import isnan, isnull
null_counts = [(c, df.filter(isnull(col(c))).count()) for c in df.columns]
for col_name, cnt in null_counts:
    if cnt > 0:
        print(f"  {col_name}: {cnt} nulls")
print("no nulls found" if all(c == 0 for _, c in null_counts) else "")

Dataset Shape
Rows: 9017, Columns: 33

Sample Records
+--------------+-------------+----+------+-----------------+-----------------------------------------+----+--------------+----------+----------------------+-------------------------------------------+----------------------------------+-------+------------------------------------+-------+--------+-----+----------+-----------+-----+------------------------+--------+--------------+-----+------------------+------------+---------------------------------------------------+--------------------------+-----------------------------------+-------------------------------------------+---------------------------+----------------+----------------+
|      STATE/UT|     DISTRICT|YEAR|MURDER|ATTEMPT TO MURDER|CULPABLE HOMICIDE NOT AMOUNTING TO MURDER|RAPE|CUSTODIAL RAPE|OTHER RAPE|KIDNAPPING & ABDUCTION|KIDNAPPING AND ABDUCTION OF WOMEN AND GIRLS|KIDNAPPING AND ABDUCTION OF OTHERS|DACOITY|PREPARATION AND ASSEMBLY FOR DACOITY|ROBBERY|BURGLARY|THEFT|AU

4- Distributed processing demo

In [ ]:
print("Spark Distributed Processing Info")
print(f"Number of Partitions (RDD): {df.rdd.getNumPartitions()}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")

# Repartition to demonstrate distributed capability
df_partitioned = df.repartition(8)
print(f"After repartition: {df_partitioned.rdd.getNumPartitions()} partitions")

# Show execution plan — this is Spark's distributed query plan
print("\nSpark Distributed Execution Plan")
df.groupBy("STATE/UT") \
  .agg(sum("TOTAL IPC CRIMES").alias("Total"))

Spark Distributed Processing Info
Number of Partitions (RDD): 1
Default Parallelism: 2
After repartition: 8 partitions

Spark Distributed Execution Plan


DataFrame[STATE/UT: string, Total: bigint]

5- Analytics (6 analyses)

In [ ]:
from pyspark.sql.functions import round

# Analysis 1: Total IPC Crimes by State
print("Total Crimes by State")
df.groupBy("STATE/UT") \
  .agg(sum("TOTAL IPC CRIMES").alias("Total_Crimes")) \
  .orderBy(desc("Total_Crimes")) \
  .show(35)

Total Crimes by State
+-----------------+------------+
|         STATE/UT|Total_Crimes|
+-----------------+------------+
|   MADHYA PRADESH|     4827540|
|      MAHARASHTRA|     4546872|
|       TAMIL NADU|     4120352|
|   ANDHRA PRADESH|     4037962|
|    UTTAR PRADESH|     3716148|
|        RAJASTHAN|     3711832|
|        KARNATAKA|     2962126|
|           KERALA|     2874918|
|          GUJARAT|     2771550|
|            BIHAR|     2692586|
|      WEST BENGAL|     2238608|
|           ODISHA|     1295892|
|         DELHI UT|     1266348|
|            ASSAM|     1195528|
|          HARYANA|     1190606|
|     CHHATTISGARH|     1122054|
|        JHARKHAND|      844702|
|           PUNJAB|      768262|
|  JAMMU & KASHMIR|      518310|
| HIMACHAL PRADESH|      309896|
|      UTTARAKHAND|      206408|
|       PUDUCHERRY|      108232|
|          TRIPURA|      105468|
|       CHANDIGARH|       81614|
|          MANIPUR|       70144|
|              GOA|       64102|
|ARUNACHAL PRADESH|  

In [ ]:
# Analysis 2: Year-wise national crime trend
print(" Year-wise Crime Trend (2001-2012) ")
df.groupBy("YEAR") \
  .agg(sum("TOTAL IPC CRIMES").alias("Total_Crimes")) \
  .orderBy("YEAR") \
  .show()

=== Year-wise Crime Trend (2001-2012) ===
+----+------------+
|YEAR|Total_Crimes|
+----+------------+
|2001|     3538616|
|2002|     3560660|
|2003|     3432240|
|2004|     3664020|
|2005|     3645204|
|2006|     3756586|
|2007|     3979346|
|2008|     4186758|
|2009|     4242690|
|2010|     4449662|
|2011|     4651150|
|2012|     4774376|
+----+------------+



In [ ]:
# Analysis 3: Crimes Against Women by State
print("Crimes Against Women by State")
df.groupBy("STATE/UT") \
  .agg(
      sum("RAPE").alias("Rape"),
      sum("DOWRY DEATHS").alias("Dowry_Deaths"),
      sum("KIDNAPPING AND ABDUCTION OF WOMEN AND GIRLS").alias("Kidnapping_Women"),
      sum("CRUELTY BY HUSBAND OR HIS RELATIVES").alias("Cruelty_By_Husband"),
      sum("ASSAULT ON WOMEN WITH INTENT TO OUTRAGE HER MODESTY").alias("Assault_Women"),
      (sum("RAPE") +
       sum("DOWRY DEATHS") +
       sum("KIDNAPPING AND ABDUCTION OF WOMEN AND GIRLS") +
       sum("CRUELTY BY HUSBAND OR HIS RELATIVES") +
       sum("ASSAULT ON WOMEN WITH INTENT TO OUTRAGE HER MODESTY")
      ).alias("Total_Women_Crimes")
  ) \
  .orderBy(desc("Total_Women_Crimes")) \
  .show()

Crimes Against Women by State
+---------------+-----+------------+----------------+------------------+-------------+------------------+
|       STATE/UT| Rape|Dowry_Deaths|Kidnapping_Women|Cruelty_By_Husband|Assault_Women|Total_Women_Crimes|
+---------------+-----+------------+----------------+------------------+-------------+------------------+
| ANDHRA PRADESH|26958|       12430|           29872|            238014|       103998|            411272|
|    WEST BENGAL|41574|       10344|           43546|            261336|        45742|            402542|
|  UTTAR PRADESH|38116|       47648|           95180|            155234|        59396|            395574|
|      RAJASTHAN|31596|       10132|           49342|            200202|        61706|            352978|
| MADHYA PRADESH|72174|       18072|           18486|             79938|       159756|            348426|
|    MAHARASHTRA|35972|        8498|           21706|            160726|        78438|            305340|
|          ASSAM

In [ ]:
# Analysis 4: Most dangerous Districts
print("Top 20 Most Dangerous Districts")

df.filter(~col("DISTRICT").like("%TOTAL%")) \
  .groupBy("STATE/UT", "DISTRICT") \
  .agg(sum("TOTAL IPC CRIMES").alias("Total_Crimes")) \
  .orderBy(desc("Total_Crimes")) \
  .show(20)

Top 20 Most Dangerous Districts
+--------------+-----------------+------------+
|      STATE/UT|         DISTRICT|Total_Crimes|
+--------------+-----------------+------------+
|     KARNATAKA| BANGALORE COMMR.|      350347|
|   MAHARASHTRA|    MUMBAI COMMR.|      222670|
|       GUJARAT| AHMEDABAD COMMR.|      218005|
|MADHYA PRADESH|           INDORE|      204398|
|ANDHRA PRADESH|   HYDERABAD CITY|      202931|
|MADHYA PRADESH|           BHOPAL|      169575|
|    TAMIL NADU|          CHENNAI|      164467|
|   WEST BENGAL|          KOLKATA|      158429|
|         BIHAR|            PATNA|      147542|
|   MAHARASHTRA|           MUMBAI|      141815|
|ANDHRA PRADESH|        CYBERABAD|      141743|
|   MAHARASHTRA|      PUNE COMMR.|      139973|
|   WEST BENGAL|24 PARGANAS NORTH|      122795|
|        KERALA|  ERNAKULAM RURAL|      122473|
|   WEST BENGAL|24 PARGANAS SOUTH|      120912|
|MADHYA PRADESH|         JABALPUR|      119446|
| UTTAR PRADESH|          LUCKNOW|      118377|
|MADHYA 

In [ ]:
# Analysis 5: Murder vs Theft comparison by State
print(" Murder vs Theft by State")
df.groupBy("STATE/UT") \
  .agg(
      sum("MURDER").alias("Total_Murders"),
      sum("THEFT").alias("Total_Thefts"),
      sum("ROBBERY").alias("Total_Robberies")
  ) \
  .orderBy(desc("Total_Murders")) \
  .show()

 Murder vs Theft by State
+---------------+-------------+------------+---------------+
|       STATE/UT|Total_Murders|Total_Thefts|Total_Robberies|
+---------------+-------------+------------+---------------+
|  UTTAR PRADESH|       130886|      624094|          61534|
|          BIHAR|        82490|      306744|          47332|
|    MAHARASHTRA|        65534|     1113614|          75438|
| ANDHRA PRADESH|        63512|      574380|          15210|
| MADHYA PRADESH|        56798|      527720|          44628|
|    WEST BENGAL|        42112|      380936|          14708|
|     TAMIL NADU|        40254|      370320|          22544|
|      KARNATAKA|        39874|      399920|          34574|
|      JHARKHAND|        38120|      142644|          17422|
|      RAJASTHAN|        31688|      465948|          18142|
|          ASSAM|        30864|      170950|          14934|
|         ODISHA|        28906|      165674|          29410|
|        GUJARAT|        27550|      426414|          28766

In [ ]:
# Analysis 6: Avg crimes per district per year
print(" Avg Total IPC Crimes per District per Year ")
df.groupBy("YEAR") \
  .agg(round(avg("TOTAL IPC CRIMES"), 2).alias("Avg_Crimes_Per_District")) \
  .orderBy("YEAR") \
  .show()

 Avg Total IPC Crimes per District per Year 
+----+-----------------------+
|YEAR|Avg_Crimes_Per_District|
+----+-----------------------+
|2001|                 4942.2|
|2002|                4952.24|
|2003|                4714.62|
|2004|                5026.09|
|2005|                4972.99|
|2006|                5076.47|
|2007|                5355.78|
|2008|                5501.65|
|2009|                5531.54|
|2010|                5712.02|
|2011|                5880.09|
|2012|                5887.02|
+----+-----------------------+



In [ ]:
df.createOrReplaceTempView("crime_data")

# Top 3 states each year by total crimes
spark.sql("""
    WITH RankedCrimes AS (
        SELECT YEAR, `STATE/UT`,
               SUM(`TOTAL IPC CRIMES`) AS Total_Crimes,
               RANK() OVER (PARTITION BY YEAR ORDER BY SUM(`TOTAL IPC CRIMES`) DESC) AS Rank
        FROM crime_data
        GROUP BY YEAR, `STATE/UT`
    )
    SELECT YEAR, `STATE/UT`, Total_Crimes, Rank
    FROM RankedCrimes
    WHERE Rank <= 3
    ORDER BY YEAR, Rank
""").show(60)

+----+--------------+------------+----+
|YEAR|      STATE/UT|Total_Crimes|Rank|
+----+--------------+------------+----+
|2001|MADHYA PRADESH|      363482|   1|
|2001| UTTAR PRADESH|      356258|   2|
|2001|   MAHARASHTRA|      342466|   3|
|2002|MADHYA PRADESH|      383598|   1|
|2002|    TAMIL NADU|      333884|   2|
|2002|   MAHARASHTRA|      330924|   3|
|2003|MADHYA PRADESH|      382156|   1|
|2003|   MAHARASHTRA|      328612|   2|
|2003|    TAMIL NADU|      314372|   3|
|2004|MADHYA PRADESH|      393734|   1|
|2004|   MAHARASHTRA|      352604|   2|
|2004|    TAMIL NADU|      333212|   3|
|2005|MADHYA PRADESH|      378344|   1|
|2005|   MAHARASHTRA|      374054|   2|
|2005|    TAMIL NADU|      324720|   3|
|2006|MADHYA PRADESH|      389422|   1|
|2006|   MAHARASHTRA|      383576|   2|
|2006|ANDHRA PRADESH|      347818|   3|
|2007|MADHYA PRADESH|      404772|   1|
|2007|   MAHARASHTRA|      391414|   2|
|2007|ANDHRA PRADESH|      350174|   3|
|2008|MADHYA PRADESH|      413112|   1|


In [ ]:
# Crime growth: 2001 vs 2012 per state
spark.sql("""
    SELECT `STATE/UT`,
           SUM(CASE WHEN YEAR = 2001 THEN `TOTAL IPC CRIMES` ELSE 0 END) AS Crimes_2001,
           SUM(CASE WHEN YEAR = 2012 THEN `TOTAL IPC CRIMES` ELSE 0 END) AS Crimes_2012,
           ROUND(
               (SUM(CASE WHEN YEAR = 2012 THEN `TOTAL IPC CRIMES` ELSE 0 END) -
                SUM(CASE WHEN YEAR = 2001 THEN `TOTAL IPC CRIMES` ELSE 0 END)) * 100.0 /
                NULLIF(SUM(CASE WHEN YEAR = 2001 THEN `TOTAL IPC CRIMES` ELSE 0 END), 0),
           2) AS Growth_Pct
    FROM crime_data
    GROUP BY `STATE/UT`
    ORDER BY Growth_Pct DESC
""").show()

+---------------+-----------+-----------+----------+
|       STATE/UT|Crimes_2001|Crimes_2012|Growth_Pct|
+---------------+-----------+-----------+----------+
|    WEST BENGAL|     123126|     322854|    162.21|
|        TRIPURA|       5602|      12528|    123.63|
|          ASSAM|      73754|     155364|    110.65|
|    LAKSHADWEEP|         72|        120|     66.67|
|          BIHAR|     176864|     293228|     65.79|
|        HARYANA|      77518|     124960|     61.20|
|      JHARKHAND|      50894|      81892|     60.91|
|            GOA|       4682|       7216|     54.12|
|         KERALA|     207694|     317978|     53.10|
|      MEGHALAYA|       3374|       5114|     51.57|
|        MANIPUR|       4978|       7474|     50.14|
| ANDHRA PRADESH|     260178|     385044|     47.99|
|         ODISHA|      93322|     135914|     45.64|
|   CHHATTISGARH|      76920|     109196|     41.96|
|     TAMIL NADU|     309602|     400948|     29.50|
|         PUNJAB|      55548|      71580|     

In [ ]:
from pyspark.ml.clustering import KMeans
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator, ClusteringEvaluator
from pyspark.ml.feature import StandardScaler
from pyspark.sql.functions import col, sum, avg, count, desc, round, when
import pandas as pd
import folium

#Step 1: Aggregate crime stats per State
state_agg = df.groupBy("STATE/UT").agg(
    sum("TOTAL IPC CRIMES").alias("Total_Crimes"),
    sum("MURDER").alias("Murder"),
    sum("RAPE").alias("Rape"),
    sum("THEFT").alias("Theft"),
    sum("ROBBERY").alias("Robbery"),
    sum("KIDNAPPING & ABDUCTION").alias("Kidnapping"),
    sum("RIOTS").alias("Riots"),
    sum("DOWRY DEATHS").alias("Dowry_Deaths"),
)

# Step 2: Assemble features for clustering
hotspot_features = ["Total_Crimes", "Murder", "Rape", "Theft",
                    "Robbery", "Kidnapping", "Riots", "Dowry_Deaths"]

hotspot_assembler = VectorAssembler(
    inputCols=hotspot_features, outputCol="raw_features"
)
hotspot_scaler = StandardScaler(
    inputCol="raw_features", outputCol="features",
    withStd=True, withMean=False
)

#Step 3: KMeans
kmeans = KMeans(featuresCol="features", k=3, seed=42, maxIter=50)

hotspot_pipeline = Pipeline(stages=[hotspot_assembler, hotspot_scaler, kmeans])
hotspot_model    = hotspot_pipeline.fit(state_agg)
clustered        = hotspot_model.transform(state_agg)

#Step 4: Silhouette Score
evaluator  = ClusteringEvaluator(featuresCol="features", metricName="silhouette")
silhouette = evaluator.evaluate(clustered)
print(f"KMeans done! Silhouette Score: {silhouette:.4f}")
print("   (Closer to 1.0 = better-defined clusters)\n")

#Step 5: Label clusters as High / Medium / Low
cluster_pdf = clustered.select(
    "STATE/UT", "Total_Crimes", "Murder", "Rape", "Theft", "prediction"
).toPandas()

cluster_means   = cluster_pdf.groupby("prediction")["Total_Crimes"].mean()
sorted_clusters = cluster_means.sort_values(ascending=False).index.tolist()
label_map = {
    sorted_clusters[0]: "HIGH RISK",
    sorted_clusters[1]: "MEDIUM RISK",
    sorted_clusters[2]: "LOW RISK"
}
cluster_pdf["Risk_Level"] = cluster_pdf["prediction"].map(label_map)

KMeans done! Silhouette Score: 0.6406
   (Closer to 1.0 = better-defined clusters)



In [ ]:
print("     PREDICTIVE CRIME HOTSPOT CLASSIFICATION")
for level in ["HIGH RISK", "MEDIUM RISK", "LOW RISK"]:
    states = cluster_pdf[cluster_pdf["Risk_Level"] == level]["STATE/UT"].tolist()
    print(f"\n{level}:")
    for s in states:
        crimes = cluster_pdf[cluster_pdf["STATE/UT"] == s]["Total_Crimes"].values[0]
        print(f"   {s:<35} Total Crimes: {int(crimes):,}")

     PREDICTIVE CRIME HOTSPOT CLASSIFICATION

HIGH RISK:
   BIHAR                               Total Crimes: 2,692,586
   UTTAR PRADESH                       Total Crimes: 3,716,148
   MADHYA PRADESH                      Total Crimes: 4,827,540
   MAHARASHTRA                         Total Crimes: 4,546,872

MEDIUM RISK:
   WEST BENGAL                         Total Crimes: 2,238,608
   RAJASTHAN                           Total Crimes: 3,711,832
   GUJARAT                             Total Crimes: 2,771,550
   KARNATAKA                           Total Crimes: 2,962,126
   DELHI UT                            Total Crimes: 1,266,348
   TAMIL NADU                          Total Crimes: 4,120,352
   HARYANA                             Total Crimes: 1,190,606
   ODISHA                              Total Crimes: 1,295,892
   ANDHRA PRADESH                      Total Crimes: 4,037,962
   KERALA                              Total Crimes: 2,874,918
   ASSAM                               Total Cr

In [ ]:
#Feature Importance from Random Forest
import pandas as pd

rf_model = model.stages[-1]

feature_names = [
    "MURDER", "RAPE", "KIDNAPPING & ABDUCTION",
    "DACOITY", "ROBBERY", "BURGLARY", "THEFT",
    "RIOTS", "HURT/GREVIOUS HURT", "DOWRY DEATHS",
    "CHEATING", "ARSON", "CAUSING DEATH BY NEGLIGENCE",
    "state_index", "YEAR"
]

importances = rf_model.featureImportances.toArray()

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
}).sort_values("Importance", ascending=False)

print(importance_df.to_string(index=False))

                    Feature  Importance
         HURT/GREVIOUS HURT    0.322249
                      THEFT    0.190956
                   BURGLARY    0.145924
                     MURDER    0.070616
                      ARSON    0.052705
                state_index    0.047152
CAUSING DEATH BY NEGLIGENCE    0.040375
                   CHEATING    0.035144
                    ROBBERY    0.030535
     KIDNAPPING & ABDUCTION    0.021928
                      RIOTS    0.016463
                       RAPE    0.010285
                    DACOITY    0.006091
               DOWRY DEATHS    0.005978
                       YEAR    0.003599


In [ ]:
#Choropleth Map
import branca.colormap as cm
import urllib.request, json, folium
from IPython.display import display

# Fetch India GeoJSON with state boundaries
geojson_url = "https://raw.githubusercontent.com/geohacker/india/master/state/india_telengana.geojson"
with urllib.request.urlopen(geojson_url) as response:
    india_geojson = json.loads(response.read())

# Map dataset state names → GeoJSON state names
name_map = {
    "ANDHRA PRADESH"      : "Andhra Pradesh",
    "ARUNACHAL PRADESH"   : "Arunachal Pradesh",
    "ASSAM"               : "Assam",
    "BIHAR"               : "Bihar",
    "CHHATTISGARH"        : "Chhattisgarh",
    "GOA"                 : "Goa",
    "GUJARAT"             : "Gujarat",
    "HARYANA"             : "Haryana",
    "HIMACHAL PRADESH"    : "Himachal Pradesh",
    "JAMMU & KASHMIR"     : "Jammu and Kashmir",
    "JHARKHAND"           : "Jharkhand",
    "KARNATAKA"           : "Karnataka",
    "KERALA"              : "Kerala",
    "MADHYA PRADESH"      : "Madhya Pradesh",
    "MAHARASHTRA"         : "Maharashtra",
    "MANIPUR"             : "Manipur",
    "MEGHALAYA"           : "Meghalaya",
    "MIZORAM"             : "Mizoram",
    "NAGALAND"            : "Nagaland",
    "ODISHA"              : "Orissa",
    "PUNJAB"              : "Punjab",
    "RAJASTHAN"           : "Rajasthan",
    "SIKKIM"              : "Sikkim",
    "TAMIL NADU"          : "Tamil Nadu",
    "TRIPURA"             : "Tripura",
    "UTTAR PRADESH"       : "Uttar Pradesh",
    "UTTARAKHAND"         : "Uttaranchal",
    "WEST BENGAL"         : "West Bengal",
    "DELHI"            : "New Delhi",
    "CHANDIGARH"          : "Chandigarh",
    "PUDUCHERRY"          : "Puducherry",
    "A & N ISLANDS"       : "Andaman and Nicobar",
    "D & N HAVELI"        : "Dadra and Nagar Haveli",
    "DAMAN & DIU"         : "Daman and Diu",
    "LAKSHADWEEP"         : "Lakshadweep",
}

# ── Build risk score lookup (numeric — needed for choropleth) ─────────
risk_score_map = {
    "HIGH RISK"   : 3,
    "MEDIUM RISK" : 2,
    "LOW RISK"    : 1
}

# Build a lookup: GeoJSON state name → risk score
geo_risk = {}
geo_info = {}
for _, row in cluster_pdf.iterrows():
    geo_name = name_map.get(row["STATE/UT"])
    if geo_name:
        geo_risk[geo_name] = risk_score_map[row["Risk_Level"]]
        geo_info[geo_name] = {
            "risk"   : row["Risk_Level"],
            "crimes" : int(row["Total_Crimes"]),
            "murder" : int(row["Murder"]),
            "rape"   : int(row["Rape"]),
            "theft"  : int(row["Theft"]),
        }

# ✅ Telangana fix — was part of Andhra Pradesh in 2001–2012
ap_row = cluster_pdf[cluster_pdf["STATE/UT"] == "ANDHRA PRADESH"].iloc[0]
geo_risk["Telangana"] = risk_score_map[ap_row["Risk_Level"]]
geo_info["Telangana"] = {
    "risk"   : ap_row["Risk_Level"] + " (was Andhra Pradesh pre-2014)",
    "crimes" : int(ap_row["Total_Crimes"]),
    "murder" : int(ap_row["Murder"]),
    "rape"   : int(ap_row["Rape"]),
    "theft"  : int(ap_row["Theft"]),
}

# ── Build Choropleth Map ──────────────────────────────────────────────
india_map = folium.Map(
    location=[22.5937, 82.9629],
    zoom_start=5,
    tiles="CartoDB positron"
)

# ✅ Green → Yellow → Orange → Red choropleth
folium.Choropleth(
    geo_data       = india_geojson,
    data           = pd.Series(geo_risk),
    key_on         = "feature.properties.NAME_1",
    fill_color     = "RdYlGn_r",
    fill_opacity   = 0.85,
    line_opacity   = 0.4,
    nan_fill_color = "lightgray",
    legend_caption = "Crime Risk Level  (1 = Low 🟢  →  3 = High 🔴)"
).add_to(india_map)

# ── Add hover tooltips on each state ─────────────────────────────────
for feature in india_geojson["features"]:
    geo_name = feature["properties"].get("NAME_1", "")
    info = geo_info.get(geo_name)
    if not info:
        continue

    folium.GeoJson(
        feature,
        style_function = lambda x: {
            "fillOpacity": 0,
            "color"      : "transparent",
            "weight"     : 0
        },
        tooltip = folium.Tooltip(
            f"<b>{geo_name}</b><br>"
            f"Risk Level   : {info['risk']}<br>"
            f"Total Crimes : {info['crimes']:,}<br>"
            f"Murder       : {info['murder']:,}<br>"
            f"Rape         : {info['rape']:,}<br>"
            f"Theft        : {info['theft']:,}"
        )
    ).add_to(india_map)

legend_html = """
<div style="position:fixed; bottom:40px; left:40px; z-index:1000;
            background:white; padding:12px 16px; border-radius:8px;
            border:2px solid #aaa; font-size:13px; font-family:Arial">
  <b>🗺️ Crime Risk Legend</b><br><br>
  <span style="background:#e74c3c;padding:2px 10px;border-radius:3px">&nbsp;</span>&nbsp; HIGH RISK<br><br>
  <span style="background:#e67e22;padding:2px 10px;border-radius:3px">&nbsp;</span>&nbsp; MEDIUM-HIGH<br><br>
  <span style="background:#f1c40f;padding:2px 10px;border-radius:3px">&nbsp;</span>&nbsp; MEDIUM RISK<br><br>
  <span style="background:#2ecc71;padding:2px 10px;border-radius:3px">&nbsp;</span>&nbsp; LOW RISK<br><br>
  <span style="background:lightgray;padding:2px 10px;border-radius:3px">&nbsp;</span>&nbsp; NO DATA<br><br>
  <i>Hover over state for details</i>
</div>
"""
india_map.get_root().html.add_child(folium.Element(legend_html))

india_map.save("/content/crime_hotspot_map.html")
print("✅ Choropleth map saved to /content/crime_hotspot_map.html")
display(india_map)

NameError: name 'cluster_pdf' is not defined